# 04 · Belt Conveyor geared drive: T1 temperature vs infrared

CORA v2.1, one controlled motor/gearbox **laboratory rig**, used as a component analogue for a geared conveyor drive reducer (not belt cracks or idlers). Target: healthy vs gear-worn (W25, W50, W75), 16 runs at four speeds; each speed is held out in turn, with one prediction per run. The sensor branch uses eight bounded T1 windows per run (about 4.3% of each CSV); the visual branch uses all 450 infrared frames.

Laboratory component rigs, not mining machines; sensor and visual branches are evaluated separately (no paired samples, no fusion); the models classify present condition, not remaining useful life or a repair date.

**Script:** `Models/belt_conveyor_cora_baseline.py`  
**Result file:** `Results/belt_conveyor_cora.json`

In [1]:
REQUIRED = ['Raw_Data/cora_gear_wear', 'Processed_Data/cora_gear_wear/t1_bounded_samples.npz']
SCRIPT = 'Models/belt_conveyor_cora_baseline.py'
JSON = 'belt_conveyor_cora.json'
ARGS = []
import contextlib
import io
import json
import os
import runpy
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

here = Path.cwd().resolve()
ROOT = next(p for p in (here, *here.parents) if (p / "Models").is_dir() and (p / "Preprocessing").is_dir())
RAW, PROCESSED, RESULTS = ROOT / "Raw_Data", ROOT / "Processed_Data", ROOT / "Results"

missing = [str(p.relative_to(ROOT)) for p in (ROOT / r for r in REQUIRED) if not p.exists()]
if missing:
    raise FileNotFoundError("Missing local data: " + ", ".join(missing) + ". Run Notebook/00_data_setup.ipynb first.")
for name in ("HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY", "http_proxy", "https_proxy", "all_proxy"):
    os.environ[name] = "http://127.0.0.1:9"  # offline guard: any network call now fails loudly
os.environ["NO_PROXY"] = ""


def run_script(relative_path, *args, tail=3):
    """Run a project script exactly as `python <script>` would and show its last output lines."""
    path, buffer, saved = ROOT / relative_path, io.StringIO(), sys.argv
    sys.argv = [path.name, *args]
    try:
        with contextlib.redirect_stdout(buffer):
            runpy.run_path(str(path), run_name="__main__")
    finally:
        sys.argv = saved
    text = buffer.getvalue().replace(str(ROOT) + os.sep, "").replace(str(ROOT), ".")
    for line in [l for l in text.splitlines() if l.strip()][-tail:]:
        print("  " + line)


METRICS = ("accuracy", "balanced_accuracy", "macro_precision", "macro_recall", "macro_f1")


def metric_table(result):
    rows = {}
    def walk(node, path):
        if isinstance(node, dict):
            found = {k: round(node[k], 3) for k in METRICS if isinstance(node.get(k), (int, float))}
            if found:
                rows[" / ".join(path) or "(top level)"] = found
            for key, value in node.items():
                if isinstance(value, dict):
                    walk(value, [*path, key])
    walk(result, [])
    return pd.DataFrame(rows).T


def load(name):
    path = RESULTS / name
    if not path.is_file():
        return None
    text = path.read_text(encoding="utf-8")
    return json.loads(text) if name.endswith(".json") else text


print(f"Repository root: {ROOT.name}/ | data present | offline guard on")

Repository root: capstone-week4/ | data present | offline guard on


In [2]:
committed = load(JSON)
run_script(SCRIPT, *ARGS)
result = load(JSON)
if JSON.endswith(".json"):
    same = committed is not None and metric_table(committed).equals(metric_table(result))
else:
    same = committed == result
print(f"\n{JSON}: {'matches the committed result' if same else 'differs from the committed result (see note above)'}")
if JSON.endswith(".json"):
    display(metric_table(result))

  Extracted run 15/16: W75_F50_S
  Extracted run 16/16: W75_F60_S
  Saved Results\belt_conveyor_cora.json, Results\belt_conveyor_sensor_visual_comparison.md, and Figures\belt_conveyor_cora_confusions.png

belt_conveyor_cora.json: matches the committed result


,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1
sensor / model,0.625,0.583,0.567,0.583,0.564
sensor / most_frequent_control,0.750,0.500,0.375,0.500,0.429
infrared / model,0.500,0.417,0.433,0.417,0.418
infrared / most_frequent_control,0.750,0.500,0.375,0.500,0.429


![Result figure](../Figures/belt_conveyor_cora_confusions.png)

*The figure shown is the committed version in `Figures/`; running this notebook regenerates it.*

**How to read this:** both learnt models score **below the 0.750 majority-class accuracy**. This is a completed, leakage-aware comparison with weak generalization, not an effective fault detector. Full write-up: [`Results/belt_conveyor_sensor_visual_comparison.md`](../Results/belt_conveyor_sensor_visual_comparison.md).